# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step tutorial for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their @id
record_sets = metadata.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', '')}")

# Optional: List fields and columns by @id for each record set
for rs in record_sets:
    print(f"\nFields for Record Set {rs.name} (@id={rs.id}):")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"  - Field Name: {field.name}\n    @id: {field.id}\n    Data Type: {field.data_type}")
    else:
        print("  No fields found.")

# Show sample record from first record set
if record_sets:
    example_rs_id = record_sets[0].id
    records_iter = dataset.records(record_set=example_rs_id)
    for i, record in enumerate(records_iter):
        print(f"Sample record from record set '{record_sets[0].name}' (@id={example_rs_id}):")
        pprint(record)
        if i >= 1:
            break

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from all record sets
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show column names for each record set DataFrame
for rs_id in record_set_ids:
    print(f"Columns for record set @id={rs_id}:")
    print(dataframes[rs_id].columns.tolist())
    print(dataframes[rs_id].head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Filter, normalize, and group by key fields
# For demonstration, select the first record set and numeric fields
main_rs_id = record_set_ids[0]
df = dataframes[main_rs_id]

# Find a numeric field (e.g., GAD-7 raw score)
numeric_field = None
for col in df.columns:
    # Attempt to guess numeric column by field name
    if ('gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower()) and 'score' in col.lower():
        numeric_field = col
        break
if numeric_field is None:
    # Fallback to first numeric column
    numeric_columns = df.select_dtypes(include=['float', 'int']).columns
    if len(numeric_columns):
        numeric_field = numeric_columns[0]

print(f"Using numeric field: {numeric_field}")

# Filter for high scores
threshold = 10

if numeric_field:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field (e.g., 'gender', 'village')
    group_fields = [col for col in df.columns if col.lower() in ['gender', 'village', 'marital_status', 'level_of_education']]
    if group_fields:
        group_field = group_fields[0]
        print(f"Grouping filtered data by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No group field found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization example: Histogram of numeric field and grouped mean bar chart
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Show mean value by group field, if available
    if group_fields:
        group_field = group_fields[0]
        plt.figure(figsize=(8,4))
        mean_scores = df.groupby(group_field)[numeric_field].mean().reset_index()
        sns.barplot(x=group_field, y=numeric_field, data=mean_scores)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides rich survey data on mental health indicators among residents of Kilifi County, Kenya.
- Numeric assessment scores (such as GAD-7, PHQ-9, PSQ) are available for EDA and visualization.
- Demographic fields (e.g., gender, education, village) allow grouping and stratified analysis.
- Data can be loaded, processed, filtered, grouped, and visualized using `mlcroissant` and standard Python scientific libraries.

Continue exploring, modeling, and visualizing the data for deeper insight into mental health trends!